In [1]:
!pip install torch nltk kaggle


   ---------- ----------------------------- 1/4 [python-slugify]
   ---------- ----------------------------- 1/4 [python-slugify]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- ------------------- 2/4 [kagglesdk]
   -------------------- -----

In [11]:
!pip install datasets==2.19.0
!pip install huggingface_hub -U

Found existing installation: datasets 4.8.4
Uninstalling datasets-4.8.4:
  Successfully uninstalled datasets-4.8.4
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/542.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/542.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/542.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/542.0 kB ? eta -:--:--
   ------------------------------------ - 524.3/542.0 kB 882.6 kB/s eta 0:00:01
   ---------------------------------------- 542.0/542.0 kB 853.3 kB/s  0:00:00

  Attempting uninstall: fsspec

    Found existing installation: fsspec 2026.2.0

    Uninstalling fsspec-2026.2.0:

      Successfully uninstalled fsspec-2026.2.0

   -------- ------------------------------- 1/5 [fsspec]
   -------- ------------------------------- 1/5 [fsspec]
   --------

In [26]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import nltk
import numpy as np
from collections import Counter

nltk.download('punkt_tab')

# =========================
# 1. LOAD SQUAD
# =========================

dataset = load_dataset("squad")

data = dataset["train"]

pairs = []

for item in data:
    q = item["question"]
    a = item["answers"]["text"][0]  # first answer
    pairs.append((q, a))

print("Total pairs:", len(pairs))

# optional speed limit
pairs = pairs[:30000]

# =========================
# 2. TOKENIZER
# =========================

def tokenize(text):
    return nltk.word_tokenize(text.lower())

# =========================
# 3. VOCAB BUILD
# =========================

counter = Counter()

for q, a in pairs:
    counter.update(tokenize(q))
    counter.update(tokenize(a))

vocab_size = 12000
most_common = counter.most_common(vocab_size - 4)

PAD = "<PAD>"
UNK = "<UNK>"
SOS = "<SOS>"
EOS = "<EOS>"

idx2word = [PAD, UNK, SOS, EOS] + [w for w, _ in most_common]
word2idx = {w: i for i, w in enumerate(idx2word)}

print("Vocab:", len(word2idx))

# =========================
# 4. ENCODE
# =========================

MAX_LEN = 20

def encode(text):
    tokens = tokenize(text)
    tokens = [SOS] + tokens + [EOS]

    ids = [word2idx.get(t, word2idx[UNK]) for t in tokens]

    if len(ids) < MAX_LEN:
        ids += [word2idx[PAD]] * (MAX_LEN - len(ids))
    else:
        ids = ids[:MAX_LEN]

    return ids

# =========================
# 5. DATASET
# =========================

class QADataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        q, a = self.pairs[idx]
        return torch.tensor(encode(q)), torch.tensor(encode(a))

dataset = QADataset(pairs)
loader = DataLoader(dataset, batch_size=32, shuffle=True) 
print(dataset.pairs[0])

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Total pairs: 87599
Vocab: 12000
('To whom did the Virgin Mary allegedly appear in 1858 in Lourdes France?', 'Saint Bernadette Soubirous')
